# Etapa (a): prevendo vencedor e placar de jogos da NBA

Este notebook explica, passo a passo, o que os scripts `variaveis.py` e `treinar_modelo.py` fazem.
Ele não tem lógica própria: usa as mesmas funções dos scripts, para que a explicação e o código
de verdade nunca fiquem diferentes.

**Antes de começar:** o banco `data/nba.db` precisa existir. Se não existir, rode
`scripts\coletar_jogos.py` primeiro.

**Roteiro**
1. Ver os dados que estão no banco
2. Transformar estatísticas brutas em eficiência (posses, ataque, defesa, ritmo)
3. Montar as variáveis sem vazamento de dados (a parte mais importante)
4. Juntar mandante e visitante em uma linha por partida
5. Separar treino e teste no tempo
6. Criar referências simples para comparar
7. Treinar os modelos
8. Avaliar
9. Interpretar o que o modelo aprendeu
10. Olhar previsões de jogos reais

## Passo 0: preparar o ambiente

Os scripts ficam na pasta `scripts`. A primeira célula avisa ao Python onde eles estão, para que
possamos importar as funções deles aqui. Ela funciona tanto se o notebook for aberto na pasta
`notebooks` quanto na raiz do projeto.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))

import numpy as np
import pandas as pd

from banco import conectar
from variaveis import (
    MIN_JOGOS_ANTES,
    VARIAVEIS_MODELO,
    carregar_time_no_jogo,
    historico_do_time,
    montar_partidas,
)
from treinar_modelo import TEMPORADAS_TESTE, TEMPORADAS_TREINO, avaliar, mostrar, treinar

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
conexao = conectar()

## Passo 1: os dados no banco

O banco tem uma linha por **time em cada partida** (tabela `team_games`), ou seja, duas linhas por
jogo. São 5 temporadas regulares, cada uma com 1.230 jogos (30 times × 82 jogos ÷ 2).

A visão `vw_jogos` já coloca mandante e visitante lado a lado. Com ela dá para ver a primeira
referência do projeto: com que frequência o time da casa vence.

In [ ]:
pd.read_sql_query(
    '''
    SELECT season AS temporada,
           COUNT(*) AS partidas,
           ROUND(100.0 * AVG(home_won), 1) AS mandante_vence_pct,
           ROUND(AVG(home_pts + away_pts), 1) AS pontos_por_jogo
    FROM vw_jogos
    GROUP BY season
    ''',
    conexao,
)

O mandante vence cerca de 55% das vezes. Então um "modelo" que sempre aposta no time da casa já
acerta 55%. **Qualquer modelo que não passe disso não serve para nada.**

## Passo 2: de estatísticas brutas para eficiência

Comparar só "pontos por jogo" engana: um time que joga rápido tem mais posses de bola e por isso
marca e sofre mais pontos, sem ser necessariamente melhor. Por isso o projeto mede tudo **por posse**.

A estimativa usual de posses de um time em uma partida é:

$$\text{posses} = \text{arremessos tentados} - \text{rebotes ofensivos} + \text{desperdícios} + 0{,}44 \times \text{lances livres tentados}$$

- Rebote ofensivo não encerra a posse, por isso é subtraído.
- O 0,44 aproxima quantos lances livres encerram uma posse (há faltas com 2 ou 3 lances e lances
  extras após cesta).

Com isso o projeto calcula, para cada time:

| Variável | Fórmula | Significado |
|---|---|---|
| ataque | 100 × pontos marcados ÷ posses | pontos a cada 100 posses |
| defesa | 100 × pontos sofridos ÷ posses | quanto menor, melhor |
| ritmo | 48 × posses ÷ minutos | posses em um jogo de 48 minutos |

A função `carregar_time_no_jogo` traz cada time com os números do adversário ao lado e já calcula as posses.

In [ ]:
df = carregar_time_no_jogo(conexao)
df[["season", "game_date", "team_id", "is_home", "won", "pts", "opp_pts", "posses", "minutos_jogo"]].head()

## Passo 3: variáveis sem vazamento de dados

Esta é a parte mais importante do projeto.

Para prever um jogo, o modelo só pode usar o que **já tinha acontecido antes dele**. Se a variável
"aproveitamento" de um jogo incluísse o resultado desse mesmo jogo, o modelo estaria colando: ele
pareceria ótimo no teste e falharia na hora de prever um jogo de verdade. Isso se chama
**vazamento de dados** (*data leakage*).

O truque usado em `historico_do_time`:

- `cumsum()` soma os jogos da temporada **até o atual, incluindo ele**;
- subtraindo o próprio jogo, sobra só o passado;
- `cumcount()` conta quantos jogos vieram antes (0 no primeiro jogo).

Para a fase recente, `shift(1)` empurra a série uma posição para baixo, de modo que cada jogo só
enxerga os 10 anteriores.

Veja os primeiros jogos de um time em 2025-26. Repare que, em cada linha, o aproveitamento é
calculado só com as linhas de cima.

In [ ]:
hist = historico_do_time(df)
um_time = df.join(hist.drop(columns=["game_id", "team_id"]))
um_time = um_time[(um_time["team_id"] == um_time["team_id"].iloc[0]) & (um_time["season"] == "2025-26")]
um_time[["game_date", "won", "jogos_antes", "aproveitamento", "ataque", "defesa", "descanso"]].head(7)

No primeiro jogo não há histórico (`jogos_antes = 0`), por isso as médias aparecem vazias (`NaN`).
Com poucos jogos as médias também oscilam muito. Por isso o projeto **descarta partidas em que algum
time tem menos de 5 jogos na temporada** (`MIN_JOGOS_ANTES`).

O descanso é o número de dias desde o jogo anterior, limitado a 7. O valor 1 significa jogos em dias
seguidos (*back-to-back*), que costumam cansar o time.

> Esse cuidado foi testado: apagando do banco todos os jogos depois de uma data, as variáveis dos
> jogos até aquela data continuam idênticas. Se houvesse vazamento, elas mudariam.

## Passo 4: uma linha por partida

O modelo precisa de **uma linha por jogo**, com o mandante e o visitante lado a lado. A função
`montar_partidas` faz essa junção, renomeia as colunas para `casa_...` e `fora_...`, tira os jogos
de quadra neutra e aplica o mínimo de 5 jogos.

In [ ]:
partidas = montar_partidas(conexao)
print(f"Partidas utilizáveis: {len(partidas)} (mínimo de {MIN_JOGOS_ANTES} jogos antes para cada time)")
print("\nO que o modelo recebe:", VARIAVEIS_MODELO)
partidas[["game_date", "casa_ataque", "casa_defesa", "fora_ataque", "fora_defesa", "casa_pts", "fora_pts", "casa_venceu"]].head()

## Passo 5: treino e teste separados no tempo

Em muitos tutoriais os dados são embaralhados e divididos ao acaso. **Aqui isso seria um erro:** o
modelo treinaria com jogos de 2025 para "prever" jogos de 2023.

Na vida real a gente só tem o passado. Então:

- **treino:** 2021-22, 2022-23 e 2023-24;
- **teste:** 2024-25 e 2025-26, que o modelo nunca vê durante o treino.

In [ ]:
treino = partidas[partidas["season"].isin(TEMPORADAS_TREINO)]
teste = partidas[partidas["season"].isin(TEMPORADAS_TESTE)].copy()
print(f"Treino: {len(treino)} partidas | Teste: {len(teste)} partidas")

## Passo 6: referências simples

Um número como "67% de acerto" sozinho não diz nada. Ele precisa ser comparado com regras simples,
que qualquer pessoa usaria sem modelo nenhum:

1. **Mandante sempre vence:** acerta cerca de 55%.
2. **Melhor campanha vence:** aposta no time com maior aproveitamento até ali. É uma referência bem
   mais difícil de bater.
3. **Para o placar:** prever que cada time fará a média de pontos dele na temporada.

In [ ]:
melhor_campanha = (teste["casa_aproveitamento"] >= teste["fora_aproveitamento"]).astype(int)
print(f"Mandante sempre vence:  {100 * teste['casa_venceu'].mean():.1f}%")
print(f"Melhor campanha vence:  {100 * (melhor_campanha == teste['casa_venceu']).mean():.1f}%")

## Passo 7: os modelos

São dois modelos, ambos lineares e fáceis de interpretar:

- **Regressão logística** (`LogisticRegression`): devolve a **chance de o mandante vencer**, entre 0 e 1.
  Acima de 0,5, a previsão é vitória do mandante.
- **Regressão Ridge** (`Ridge`): prevê **os pontos de cada time**. É uma regressão linear com uma
  pequena penalidade que evita pesos exagerados quando as variáveis são parecidas entre si.

Antes dos dois entra o `StandardScaler`, que coloca todas as variáveis na mesma escala (média 0 e
desvio 1). Sem ele, uma variável medida em centenas, como o ataque, e outra em unidades, como o
descanso, receberiam pesos difíceis de comparar.

O `make_pipeline` junta o padronizador e o modelo em uma peça só. Assim a padronização aprende só com
o treino e é aplicada igual no teste.

In [ ]:
modelo_vencedor, modelo_placar = treinar(treino)

teste["prob_casa"] = modelo_vencedor.predict_proba(teste[VARIAVEIS_MODELO])[:, 1]
teste[["prev_casa_pts", "prev_fora_pts"]] = modelo_placar.predict(teste[VARIAVEIS_MODELO])
teste[["game_date", "prob_casa", "prev_casa_pts", "prev_fora_pts", "casa_pts", "fora_pts"]].head()

## Passo 8: avaliação

Três tipos de medida:

- **Acerto do vencedor:** porcentagem de jogos em que o lado previsto venceu.
- **Log loss:** avalia a *chance* prevista, não só o acerto. Dizer 90% e errar custa muito mais do que
  dizer 55% e errar. Quanto menor, melhor. A referência é dar a todo jogo a mesma chance: a taxa de
  vitória do mandante no treino.
- **Erro médio absoluto (MAE):** em média, por quantos pontos a previsão errou o placar real de cada time.

In [ ]:
taxa_mandante = treino["casa_venceu"].mean()
print(f"Taxa de vitória do mandante no treino: {taxa_mandante:.3f}")
grupos = {temporada: teste[teste["season"] == temporada] for temporada in TEMPORADAS_TESTE}
grupos["Total"] = teste
tabela = pd.DataFrame({nome: avaliar(jogos, taxa_mandante) for nome, jogos in grupos.items()})
mostrar(tabela, [len(jogos) for jogos in grupos.values()])

**Como ler:**

- O modelo bate com folga o "mandante sempre vence".
- Contra "melhor campanha vence", a vantagem é pequena. Faz sentido: o aproveitamento já resume
  boa parte da força de um time. O ganho maior aparece no **log loss**, porque o modelo diz *quão
  provável* é a vitória, e a referência não.
- Errar o placar de cada time em cerca de 9,5 pontos parece muito, mas um mesmo time costuma variar
  mais de 10 pontos de um jogo para outro. O modelo ainda assim erra menos que a média da temporada.

## Passo 9: o que o modelo aprendeu

Como as variáveis foram padronizadas, os pesos da regressão logística podem ser comparados entre si.
**Peso positivo aumenta a chance do mandante; negativo diminui.**

In [ ]:
pesos = pd.Series(modelo_vencedor[-1].coef_[0], index=VARIAVEIS_MODELO).sort_values()
pesos.round(2).to_frame("peso")

Os sinais fazem sentido no basquete:

- `casa_ataque` positivo e `casa_defesa` negativo: um mandante que marca muito e sofre pouco vence mais.
- `fora_ataque` negativo e `fora_defesa` positivo: o mesmo raciocínio, ao contrário, para o visitante.
- `casa_descanso` positivo e `fora_descanso` negativo: o time mais descansado leva vantagem.

Se algum sinal saísse ao contrário do esperado, seria um bom motivo para desconfiar dos dados ou das
variáveis.

## Passo 10: previsões de jogos reais

Para fechar, alguns jogos do fim da última temporada, com a sigla dos times, a previsão e o resultado.
Em todas as colunas o mandante vem primeiro: "LAL x UTA" e "125 x 114" significam Lakers em casa,
com 125 pontos.

In [ ]:
siglas = pd.read_sql_query("SELECT team_id, abbreviation FROM teams", conexao).set_index("team_id")["abbreviation"]
exemplo = teste.tail(8).assign(
    jogo=lambda t: t["casa_team_id"].map(siglas) + " x " + t["fora_team_id"].map(siglas),
    chance_casa=lambda t: (100 * t["prob_casa"]).round(0).astype(int).astype(str) + "%",
    previsto=lambda t: t["prev_casa_pts"].round(0).astype(int).astype(str) + " x " + t["prev_fora_pts"].round(0).astype(int).astype(str),
    real=lambda t: t["casa_pts"].astype(str) + " x " + t["fora_pts"].astype(str),
    acertou=lambda t: np.where((t["prob_casa"] >= 0.5).astype(int) == t["casa_venceu"], "sim", "não"),
)
exemplo[["game_date", "jogo", "chance_casa", "previsto", "real", "acertou"]]

O placar previsto fica sempre perto da média da liga: no teste, metade das previsões ficou entre 111
e 120 pontos, e nenhuma passou de 133. Isso é esperado num modelo linear, que prevê o resultado
**médio** de um confronto, não as surpresas. Jogos de 140 pontos acontecem, mas não dá para
prevê-los com estas variáveis.

## Próximos passos

Ideias para melhorar a etapa (a), da mais simples para a mais trabalhosa:

1. **Início de temporada:** hoje os 5 primeiros jogos de cada time ficam de fora. Daria para usar os
   números da temporada anterior como ponto de partida.
2. **Força dos adversários:** um time pode ter bom aproveitamento por ter enfrentado só times fracos.
3. **Modelos não lineares:** testar árvores de decisão e *gradient boosting*, sempre comparando com
   as mesmas referências.

In [ ]:
conexao.close()